In [17]:
import pandas as pd
import anndata as ad 
import scanpy as sc 
import os
import pickle as pkl 

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

In [20]:
paired_contrast = [
    ("Control_Young","Trained_Young"),
    ("Control_Young","Control_Old"),
    ("Control_Old","Trained_Old"),
    ("Trained_Young","Trained_Old")
]

In [18]:
adata = ad.read("./data/adata_filter_genes.h5ad")

In [19]:
adata

AnnData object with n_obs × n_vars = 16 × 22289
    obs: 'treatment', 'stimulation', 'Age', 'condition'
    var: 'gene_name'

Create your contrast here. First paired will ALWAYS be the reference!!

In [27]:
def create_model(
    adata,
    variable_of_interest,
    reference_level,
    test_level,
    prefix="",
    batch_name="",
    design="~condition",
    overwrite=False,
):
    # -----------------------------------------------------------
    # 1. Build output names
    # -----------------------------------------------------------
    #design_name = design.replace("~", "").replace(" ", "_")
    design_name = design
    model_path = os.path.join(
        "./results/models/",
        f"model_{prefix}{design_name}.pkl",
    )
    excel_path = (
        f"./results/contrasts/"
        f"{prefix}{batch_name}{variable_of_interest}_"
        f"{test_level}_VS_{reference_level}.csv"
    )

    # -----------------------------------------------------------
    # 2. Load or compute model
    # -----------------------------------------------------------
    if os.path.exists(model_path) and not overwrite:
        print(f"⚠️ Model already exists, loading: {model_path}")
        with open(model_path, "rb") as f:
            dds = pkl.load(f)

    else:
        print(f"Creating DESeq2 model for design {design}...")
        dds = DeseqDataSet(
            adata=adata,
            design=design,
            refit_cooks=True,
        )
        dds.deseq2()
        dds.vst()

        dds.plot_dispersions(
            save_path=f"./results/QC/{prefix}dispersion_{design_name}.png"
        )

        os.makedirs("./results/models/", exist_ok=True)
        with open(model_path, "wb") as f:
            pkl.dump(dds, f)

        print(f"✅ Model saved to {model_path}")

    # -----------------------------------------------------------
    # 3. Run stats for the requested contrast
    # -----------------------------------------------------------
    print(
        f"Running DESeqStats for contrast: "
        f"{variable_of_interest} {test_level} vs {reference_level}"
    )

    stat_res = DeseqStats(
        dds,
        contrast=[variable_of_interest, test_level, reference_level],
    )
    stat_res.summary()

    df_res = stat_res.results_df
    genes = adata.var
    concat = pd.concat([genes, df_res], axis=1)

    os.makedirs("./results/contrasts/", exist_ok=True)
    concat.to_csv(excel_path, index=True,sep=';')

    print(f"📁 Results saved to {excel_path}")


In [28]:
for ref,interest in paired_contrast:
    print(ref)
    adata_to_model = adata.copy()
    create_model(adata_to_model,variable_of_interest='condition',reference_level=ref,test_level=interest)

Control_Young
⚠️ Model already exists, loading: ./results/models/model_~condition.pkl
Running DESeqStats for contrast: condition Trained_Young vs Control_Young


Running Wald tests...
... done in 1.41 seconds.



Log2 fold change & Wald test p-value: condition Trained_Young vs Control_Young
                       baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                         
ENSMUSG00000000001  6614.435806        0.071631  0.061170  1.171024  0.241589   
ENSMUSG00000000028  2677.765983       -0.053403  0.067646 -0.789450  0.429849   
ENSMUSG00000000031     3.545939        0.756689  0.707121  1.070097  0.284575   
ENSMUSG00000000037   506.453130       -0.430428  0.134006 -3.211999  0.001318   
ENSMUSG00000000056  2833.805782       -0.379488  0.161120 -2.355315  0.018507   
...                         ...             ...       ...       ...       ...   
ENSMUSG00002076556    16.984368       -0.763153  0.990742 -0.770284  0.441131   
ENSMUSG00002076601     0.616981       -0.113076  2.031281 -0.055667  0.955607   
ENSMUSG00002076665     0.485802        1.202252  2.602915  0.461887  0.644162   
ENSMUSG00002076675     0.72981

Running Wald tests...
... done in 1.22 seconds.



Log2 fold change & Wald test p-value: condition Control_Old vs Control_Young
                       baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                         
ENSMUSG00000000001  6614.435806        0.075209  0.061263  1.227632  0.219585   
ENSMUSG00000000028  2677.765983       -0.051781  0.067850 -0.763160  0.445368   
ENSMUSG00000000031     3.545939        0.109256  0.764522  0.142908  0.886363   
ENSMUSG00000000037   506.453130       -0.380692  0.134451 -2.831465  0.004634   
ENSMUSG00000000056  2833.805782        0.078905  0.161023  0.490022  0.624118   
...                         ...             ...       ...       ...       ...   
ENSMUSG00002076556    16.984368       -0.499397  0.990152 -0.504364  0.614005   
ENSMUSG00002076601     0.616981        0.053467  2.036356  0.026256  0.979053   
ENSMUSG00002076665     0.485802       -0.620040  2.841000 -0.218247  0.827237   
ENSMUSG00002076675     0.729818 

Running Wald tests...
... done in 1.01 seconds.



Log2 fold change & Wald test p-value: condition Trained_Old vs Control_Old
                       baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                         
ENSMUSG00000000001  6614.435806        0.127988  0.061168  2.092398  0.036403   
ENSMUSG00000000028  2677.765983       -0.168258  0.068056 -2.472333  0.013423   
ENSMUSG00000000031     3.545939        0.312199  0.732701  0.426094  0.670039   
ENSMUSG00000000037   506.453130       -0.261740  0.136070 -1.923568  0.054409   
ENSMUSG00000000056  2833.805782       -0.576188  0.161221 -3.573906  0.000352   
...                         ...             ...       ...       ...       ...   
ENSMUSG00002076556    16.984368       -1.205691  1.012166 -1.191200  0.233575   
ENSMUSG00002076601     0.616981       -1.387464  2.275338 -0.609784  0.542005   
ENSMUSG00002076665     0.485802       -0.858608  3.142823 -0.273196  0.784702   
ENSMUSG00002076675     0.729818   

Running Wald tests...
... done in 1.51 seconds.



Log2 fold change & Wald test p-value: condition Trained_Old vs Trained_Young
                       baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_id                                                                         
ENSMUSG00000000001  6614.435806        0.131566  0.061075  2.154175  0.031226   
ENSMUSG00000000028  2677.765983       -0.166635  0.067853 -2.455834  0.014056   
ENSMUSG00000000031     3.545939       -0.335233  0.672590 -0.498421  0.618187   
ENSMUSG00000000037   506.453130       -0.212003  0.135631 -1.563087  0.118032   
ENSMUSG00000000056  2833.805782       -0.117796  0.161318 -0.730211  0.465261   
...                         ...             ...       ...       ...       ...   
ENSMUSG00002076556    16.984368       -0.941936  1.012743 -0.930083  0.352328   
ENSMUSG00002076601     0.616981       -1.220922  2.270797 -0.537662  0.590810   
ENSMUSG00002076665     0.485802       -2.680900  2.929375 -0.915178  0.360098   
ENSMUSG00002076675     0.729818 